# Build the Sinhala Diachronic POS & Lemma Dataset

Builds a field-conforming, shareable release from the components already
computed elsewhere in this repo (`Data Processing/Data_Partitioning.ipynb`,
`POS_Tagging/Merge_POS.ipynb`, `Morphological Analysis/sinmorphy_automation_v6.ipynb`,
`POS_Tagging/sinling_pos_tagger.ipynb`, `POS_Tagging/TnT_Tagger_Robust_Alignment.ipynb`),
improving on the earlier flat CSV in four ways:

1. **Keeps each tagger's individual vote** (SinMorphy, sinling, TnT) alongside the finalized value, instead of only exposing the merged result.
2. **Adds a UPOS column** (Universal Dependencies' universal POS tagset) alongside the original SinMorphy-style tag, non-destructively.
3. **Exports CoNLL-U Plus**, the standard format for POS/lemma-annotated corpora in this field, alongside a rich companion CSV.
4. **Joins document-level provenance** (title/author/genre/year) onto every token.

Outputs:
- `data/sinhala_diachronic_pos_lemma.csv`
- `data/sinhala_diachronic_pos_lemma.conllu`

In [ ]:
import os
import re
import json

import numpy as np
import pandas as pd

REPO_DATA_DIR = "../Data"
OCR_ROOT = f"{REPO_DATA_DIR}/OCR_Final"
OUT_DIR = "data"

## Step 1 — Recover document provenance per sentence

`Data/century_tokens.json` was built by `Data Processing/Data_Partitioning.ipynb`
concatenating each document's sentences into a per-century list, with document
boundaries **not preserved** (the separator token in that notebook is disabled).
To recover title/author/genre per sentence, we re-derive the same per-document
structure here.

**Important finding:** the current `.txt` files in `Data/OCR_Final/` do not
byte-for-byte reproduce `century_tokens.json` for every document — a small
number of sentences differ by a token or two (verified: both files entered the
repo in the same commit and neither has changed since, so this is a
pre-existing snapshot/source mismatch, not something introduced here). Per-token
content matching is therefore unreliable as a way to find document boundaries.

Instead we rely on what's actually guaranteed: documents are concatenated
**in order**, each contributing a **contiguous** block of sentences equal to its
own sentence count. So each document's boundaries are just the cumulative
offsets of sentence counts in traversal order — the exact content of
individual sentences doesn't need to match for this to be correct. We verify
the partition is exact (total sentences accounted for, no gaps) per century,
and additionally check the first/last sentence of each document as an
informational anchor (not required for correctness).

In [ ]:
records = []
for dirpath, dirnames, filenames in os.walk(OCR_ROOT):
    metadata_filepath = None
    non_json_files = []
    for filename in filenames:
        if filename.lower() == "metadata.json":
            metadata_filepath = os.path.join(dirpath, filename)
        elif filename.lower().endswith(".txt"):
            non_json_files.append(os.path.join(dirpath, filename))
    if metadata_filepath and len(non_json_files) == 1:
        with open(metadata_filepath, "r", encoding="utf-8") as f:
            metadata = json.load(f)
        records.append({
            "doc_id": os.path.basename(dirpath),
            "text_file_location": non_json_files[0],
            "written_date": metadata.get("written_date"),
            "title": metadata.get("title"),
            "title_en": metadata.get("title_en"),
            "author": metadata.get("author"),
            "genre": metadata.get("genre"),
            "issued_date": metadata.get("issued_date"),
            "ocr_confidence": metadata.get("ocr_confidence"),
        })

docs_df = pd.DataFrame(records)
docs_df["written_date"] = docs_df["written_date"].replace(["", None], np.nan)
docs_df.dropna(subset=["written_date"], inplace=True)
docs_df.reset_index(drop=True, inplace=True)

starts, ends = [], []
for _, row in docs_df.iterrows():
    date_str = str(row["written_date"])
    if " - " in date_str:
        s, e = date_str.split(" - ")
        starts.append(int(s.strip())); ends.append(int(e.strip()))
    else:
        y = int(date_str.strip())
        starts.append(y); ends.append(y)
docs_df["start_date"] = starts
docs_df["end_date"] = ends
docs_df["end_century"] = docs_df["end_date"].apply(lambda y: (int(y) - 1) // 100 + 1)

len(docs_df)

In [ ]:
def clean_and_tokenize_sentences_fixed(path):
    """Ported verbatim from Data Processing/Data_Partitioning.ipynb."""
    with open(path, "r", encoding="utf-8") as file:
        text = file.read()
    tokens = re.split(r"(\s+|<psi>|<eos>)", text)
    tokens = [t.strip() for t in tokens if t.strip()]
    processed_tokens = []
    i = 0
    while i < len(tokens):
        if tokens[i] == "<psi>":
            if i > 0 and i < len(tokens) - 1:
                processed_tokens[-1] = processed_tokens[-1] + tokens[i + 1]
                i += 2
            else:
                i += 1
        else:
            processed_tokens.append(tokens[i])
            i += 1
    sinhala_regex = re.compile(r"[^\u0D80-\u0DFF\s]")
    final_tokens = []
    for token in processed_tokens:
        if token == "<eos>":
            final_tokens.append(token)
        else:
            cleaned = sinhala_regex.sub("", token)
            if cleaned:
                final_tokens.append(cleaned)
    processed_text_cleaned = " ".join(final_tokens).replace("<psi>", " ")
    sentence_strings = processed_text_cleaned.split("<eos>")
    out = []
    for sentence in sentence_strings:
        toks = sentence.strip().split()
        if toks:
            out.append(toks)
    return out

docs_df["tokens"] = docs_df["text_file_location"].apply(clean_and_tokenize_sentences_fixed)
docs_df["num_sentences"] = docs_df["tokens"].apply(len)

with open(f"{REPO_DATA_DIR}/century_tokens.json", "r", encoding="utf-8") as f:
    saved_century_tokens = json.load(f)

In [ ]:
doc_meta_rows = []
anchor_mismatches = 0

for century, group in docs_df.groupby("end_century"):
    century_key = str(int(century))
    haystack = saved_century_tokens[century_key]
    offset = 0
    for _, doc_row in group.iterrows():
        n = doc_row["num_sentences"]
        start, end = offset, offset + n
        if n > 0:
            first_ok = haystack[start] == doc_row["tokens"][0]
            last_ok = haystack[end - 1] == doc_row["tokens"][-1]
            if not (first_ok and last_ok):
                anchor_mismatches += 1
                print(f"  [anchor drift, informational only] century {century_key} "
                      f"doc={doc_row['doc_id']!r} sentences=[{start},{end}) "
                      f"first_ok={first_ok} last_ok={last_ok}")
        for local_idx in range(n):
            doc_meta_rows.append({
                "century": int(century),
                "sentence_index": start + local_idx,
                "doc_id": doc_row["doc_id"],
                "doc_title": doc_row["title"],
                "doc_title_en": doc_row["title_en"],
                "doc_author": doc_row["author"],
                "doc_genre": doc_row["genre"],
                "written_date_start": doc_row["start_date"],
                "written_date_end": doc_row["end_date"],
                "issued_date": doc_row["issued_date"],
                "ocr_confidence": doc_row["ocr_confidence"],
            })
        offset = end
    assert offset == len(haystack), f"incomplete coverage for century {century_key}"

sent_meta_df = pd.DataFrame(doc_meta_rows)
total_saved_sentences = sum(len(v) for v in saved_century_tokens.values())
print(f"\nanchor mismatches (informational, does not affect coverage): {anchor_mismatches}")
print(f"sentence-level metadata rows: {len(sent_meta_df)} / {total_saved_sentences} total sentences")
assert len(sent_meta_df) == total_saved_sentences, "every sentence must get a document"

## Step 2 — Load the three POS/lemma annotation sources and finalize

Ported from `POS_Tagging/Finalize_POS_Lemma.ipynb`, but this time the
individual tagger columns (`sinmorphy_pos`, `sinmorphy_lemma`, `sinling_pos`,
`tnt_pos`, and SinMorphy's raw candidate lists) are **kept**, not dropped,
so downstream users can audit or re-resolve the ensemble decision themselves.

In [ ]:
unparsed_mask = sinmorphy_df["all_morphologies"].isna()
empty_lemma_mask = sinmorphy_df["final_lemma"].astype(str).str.strip() == ""
fallback_mask = unparsed_mask | empty_lemma_mask
print(f"Unparsed tokens: {unparsed_mask.sum()} / {len(sinmorphy_df)}")
print(f"Empty-stem morphology tokens: {empty_lemma_mask.sum()} / {len(sinmorphy_df)}")
sinmorphy_df.loc[fallback_mask, "final_lemma"] = sinmorphy_df.loc[fallback_mask, "word"]
# final_morphology inherits the same "nan"-string / degenerate-stem problem for
# these rows (there's genuinely no morphology to report) - represent as a true
# missing value rather than the literal text "nan" leaking through to CSV/CoNLL-U.
sinmorphy_df.loc[fallback_mask, "final_morphology"] = np.nan

sinmorphy_df["token_index"] = sinmorphy_df.groupby(["century", "sentence_index"]).cumcount()

sinmorphy_df["final_pos"].value_counts()

In [ ]:
def finalize_all_columns(row: pd.Series) -> pd.Series:
    """Ported verbatim from POS_Tagging/Merge_POS.ipynb."""
    all_tags_list = [t.strip() for t in str(row["all_tags"]).split("|")]
    all_morphs_list = [m.strip() for m in str(row["all_morphologies"]).split("|")]

    selected_index = 0
    if len(all_tags_list) > 1:
        tnt_res = row["tnt_pos"]
        sinling_res = row["sinling_pos"]
        if tnt_res != "UNKNOWN" and tnt_res in all_tags_list:
            selected_index = all_tags_list.index(tnt_res)
        elif sinling_res != "UNKNOWN" and sinling_res in all_tags_list:
            selected_index = all_tags_list.index(sinling_res)

    final_tag = all_tags_list[selected_index]
    try:
        final_morph = all_morphs_list[selected_index]
    except IndexError:
        final_morph = all_morphs_list[0] if all_morphs_list else "UNKNOWN"

    if "+" in final_morph:
        final_lemma = final_morph.split("+")[0]
    else:
        final_lemma = final_morph

    return pd.Series([final_tag, final_morph, final_lemma])

sinmorphy_df[["final_pos", "final_morphology", "final_lemma"]] = sinmorphy_df.apply(
    finalize_all_columns, axis=1
)

# Fix the two CSV-round-trip lemma-extraction edge cases documented in
# POS_Tagging/Finalize_POS_Lemma.ipynb: unparsed tokens (all_morphologies is
# NaN) and degenerate morphology strings with no lemma stem before the first
# '+'. Both fall back to lemma = word, matching new_lemma's existing fallback.
unparsed_mask = sinmorphy_df["all_morphologies"].isna()
empty_lemma_mask = sinmorphy_df["final_lemma"].astype(str).str.strip() == ""
fallback_mask = unparsed_mask | empty_lemma_mask
print(f"Unparsed tokens: {unparsed_mask.sum()} / {len(sinmorphy_df)}")
print(f"Empty-stem morphology tokens: {empty_lemma_mask.sum()} / {len(sinmorphy_df)}")
sinmorphy_df.loc[fallback_mask, "final_lemma"] = sinmorphy_df.loc[fallback_mask, "word"]

sinmorphy_df["token_index"] = sinmorphy_df.groupby(["century", "sentence_index"]).cumcount()

sinmorphy_df["final_pos"].value_counts()

## Step 3 — Add a UPOS column (Universal Dependencies tagset)

Non-destructive: the original SinMorphy-style finalized tag is kept as `pos`
(this dataset's XPOS-equivalent); `upos` is derived from it via a fixed
mapping, built from the same 8-category coarse scheme already used in
`POS_Tagging/sinling_pos_tagger.ipynb`.

In [ ]:
POS_TO_UPOS = {
    "NOUN": "NOUN",
    "PROPER NOUN": "PROPN",
    "Proper noun or Invalid": "X",  # unparsed / OOV tokens (no SinMorphy analysis)
    "Verb": "VERB",
    "PART": "PART",
    "ADJECTIVE": "ADJ",
    "PRONOUN": "PRON",
    "FOREIGN WORD": "X",
}

sinmorphy_df["upos"] = sinmorphy_df["final_pos"].map(POS_TO_UPOS)
assert sinmorphy_df["upos"].isna().sum() == 0, "unmapped pos category found"
sinmorphy_df["upos"].value_counts()

## Step 4 — Join document metadata onto every token

In [ ]:
merged_df = sinmorphy_df.merge(sent_meta_df, on=["century", "sentence_index"], how="left")
assert merged_df["doc_id"].isna().sum() == 0, "some tokens failed to join document metadata"
len(merged_df)

## Step 5 — Assemble and save the rich companion CSV

In [ ]:
os.makedirs(OUT_DIR, exist_ok=True)

rich_df = merged_df.rename(columns={
    "final_lemma": "lemma",
    "final_pos": "pos",
    "final_morphology": "sinmorphy_morphology",
    "pos": "sinmorphy_pos",
    "new_lemma": "sinmorphy_lemma",
    "all_tags": "sinmorphy_all_tags",
    "all_morphologies": "sinmorphy_all_morphologies",
})[[
    "century", "sentence_index", "token_index", "word", "lemma", "pos", "upos",
    "sinmorphy_pos", "sinmorphy_lemma", "sinmorphy_morphology",
    "sinmorphy_all_tags", "sinmorphy_all_morphologies",
    "sinling_pos", "tnt_pos",
    "doc_id", "doc_title", "doc_title_en", "doc_author", "doc_genre",
    "written_date_start", "written_date_end", "issued_date", "ocr_confidence",
]]

csv_path = f"{OUT_DIR}/sinhala_diachronic_pos_lemma.csv"
rich_df.to_csv(csv_path, index=False, encoding="utf-8")
print(f"Wrote {len(rich_df)} rows to {csv_path}")
rich_df.head()

## Step 6 — Export CoNLL-U Plus

Sentence-constant metadata (century, doc id/title/author/genre, written date)
goes in `#`-prefixed sentence comments, per CoNLL-U convention, rather than
being repeated on every token line. Per-token tagger votes are carried as
extra columns declared via `# global.columns`, which CoNLL-U Plus supports.
`FEATS`/`HEAD`/`DEPREL`/`DEPS`/`MISC` are unspecified (`_`) — no morphological
feature mapping or dependency parse is available in this pipeline.

In [ ]:
GLOBAL_COLUMNS = (
    "ID FORM LEMMA UPOS XPOS FEATS HEAD DEPREL DEPS MISC "
    "SinMorphyPOS SinMorphyLemma SinMorphyMorphology "
    "SinMorphyAllTags SinMorphyAllMorphologies SinlingPOS TntPOS"
)

def safe_field(value) -> str:
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return "_"
    text = str(value).strip().replace("\t", " ").replace("\n", " ")
    return text if text else "_"

conllu_path = f"{OUT_DIR}/sinhala_diachronic_pos_lemma.conllu"

with open(conllu_path, "w", encoding="utf-8") as f:
    f.write(f"# global.columns = {GLOBAL_COLUMNS}\n")
    for (century, sentence_index), sent_df in rich_df.groupby(["century", "sentence_index"], sort=True):
        sent_df = sent_df.sort_values("token_index")
        first = sent_df.iloc[0]
        written_date = (
            str(int(first["written_date_start"]))
            if first["written_date_start"] == first["written_date_end"]
            else f"{int(first['written_date_start'])}-{int(first['written_date_end'])}"
        )
        f.write(f"# sent_id = {century}_{sentence_index}\n")
        f.write(f"# text = {' '.join(sent_df['word'].astype(str))}\n")
        f.write(f"# century = {century}\n")
        f.write(f"# doc_id = {safe_field(first['doc_id'])}\n")
        f.write(f"# doc_title = {safe_field(first['doc_title'])}\n")
        f.write(f"# doc_author = {safe_field(first['doc_author'])}\n")
        f.write(f"# written_date = {written_date}\n")
        for i, (_, tok) in enumerate(sent_df.iterrows(), start=1):
            fields = [
                str(i),
                safe_field(tok["word"]),
                safe_field(tok["lemma"]),
                safe_field(tok["upos"]),
                safe_field(tok["pos"]),
                "_", "_", "_", "_", "_",
                safe_field(tok["sinmorphy_pos"]),
                safe_field(tok["sinmorphy_lemma"]),
                safe_field(tok["sinmorphy_morphology"]),
                safe_field(tok["sinmorphy_all_tags"]),
                safe_field(tok["sinmorphy_all_morphologies"]),
                safe_field(tok["sinling_pos"]),
                safe_field(tok["tnt_pos"]),
            ]
            f.write("\t".join(fields) + "\n")
        f.write("\n")

print(f"Wrote CoNLL-U Plus file to {conllu_path}")

## Step 7 — Validation

In [ ]:
print("rows in rich CSV:", len(rich_df))
print("blank lemma cells:", rich_df["lemma"].isna().sum())
print("blank pos cells:", rich_df["pos"].isna().sum())
print("blank upos cells:", rich_df["upos"].isna().sum())
print("blank doc_id cells:", rich_df["doc_id"].isna().sum())
print()
print("documents represented:", rich_df["doc_id"].nunique())
print()
print("pos -> upos mapping:")
print(rich_df.groupby("pos")["upos"].first())
print()

with open(conllu_path, "r", encoding="utf-8") as f:
    conllu_text = f.read()
num_sent_ids = conllu_text.count("# sent_id")
print("sentences written to .conllu:", num_sent_ids)
print("sentences expected (unique century/sentence_index pairs):",
      rich_df.groupby(["century", "sentence_index"]).ngroups)